# nb_ingest_pib_ibge — Ingestão de Dados de PIB (Fabric)

**Fonte:** IBGE SIDRA Tabela 5938  
**Escopo:** Santos, Osasco, Mauá e clusters comparativos.

In [ ]:
%run ./nb_utils_ibge

## 1. Bronze Layer — Ingestão

In [ ]:
TABLE_PIB = "5938"
VAR_TOTAL = "37"   # PIB Total a preços correntes
VAR_COMP  = "513,517,6575,525,543" # Componentes VAB

print("=== Ingestão PIB Total ===")
df_total_raw = fetch_sidra_fabric(TABLE_PIB, VAR_TOTAL)

print("=== Ingestão PIB Componentes ===")
df_comp_raw = fetch_sidra_fabric(TABLE_PIB, VAR_COMP)

if df_total_raw and df_comp_raw:
    save_delta(df_total_raw, "bronze_ibge_pib_total_raw")
    save_delta(df_comp_raw, "bronze_ibge_pib_componentes_raw")

## 2. Silver Layer — Padronização

In [ ]:
from pyspark.sql.functions import col, lit, trim, regexp_replace
from functools import reduce
from pyspark.sql import DataFrame

_MAP_COMP = {
    "513":  "vab_agropecuaria_r_mil",
    "517":  "vab_industria_r_mil",
    "6575": "vab_servicos_r_mil",
    "525":  "vab_adm_publica_r_mil",
    "543":  "impostos_liquidos_r_mil",
}

# Layout SIDRA t/5938 sem classificação:
# D1C/D1N = Município · D2C/D2N = Variável (37 ou 513...) · D3C/D3N = Ano · V = Valor

def standardize_pib(df, indicador_nome):
    return (
        df.select(
            col("D1C").cast("int").alias("id_municipio"),
            trim(regexp_replace(col("D1N"), r"\s*\([A-Z]{2}\)$", "")).alias("nome_municipio"),
            col("D3C").cast("int").alias("ano"),
            lit(indicador_nome).alias("indicador"),
            col("V").cast("double").alias("valor"),
        )
        .filter(col("valor").isNotNull())
    )

def standardize_pib_multi(df, map_var):
    """D2C = código da variável; D3C = ano."""
    partes = []
    for cod, indicador in map_var.items():
        parte = (
            df.filter(col("D2C") == cod)
            .select(
                col("D1C").cast("int").alias("id_municipio"),
                trim(regexp_replace(col("D1N"), r"\s*\([A-Z]{2}\)$", "")).alias("nome_municipio"),
                col("D3C").cast("int").alias("ano"),
                lit(indicador).alias("indicador"),
                col("V").cast("double").alias("valor"),
            )
            .filter(col("valor").isNotNull())
        )
        partes.append(parte)
    return reduce(DataFrame.union, partes)

# ── Silver PIB Total ──────────────────────────────────────────────────────────
if df_total_raw is not None:
    df_pib_total = standardize_pib(df_total_raw, "pib_total_r_mil")

    try:
        # Cast explícito de ano para INT — silver_populacao armazena como VARCHAR no Fabric
        df_pop = spark.sql("""
            SELECT id_municipio, CAST(ano AS INT) AS ano, valor AS populacao
            FROM silver_populacao
            WHERE indicador = 'populacao_residente'
        """)

        df_pib_pc = (
            df_pib_total
            .join(df_pop, on=["id_municipio", "ano"], how="left")
            .withColumn("indicador", lit("pib_per_capita_r"))
            .withColumn("valor", (col("valor") * 1000) / col("populacao"))
            .drop("populacao")
            .filter(col("valor").isNotNull())
            # Select explícito garante ordem e tipos idênticos ao df_pib_total antes do union
            .select(
                col("id_municipio").cast("int"),
                col("nome_municipio").cast("string"),
                col("ano").cast("int"),
                col("indicador").cast("string"),
                col("valor").cast("double"),
            )
        )
        df_silver_pib = df_pib_total.union(df_pib_pc)
        print(f"[OK] PIB per capita calculado — {df_pib_pc.count()} registros")
    except Exception as e:
        df_silver_pib = df_pib_total
        print(f"[AVISO] silver_populacao indisponivel — per capita omitido ({e})")

    save_delta(df_silver_pib, "silver_pib")
    display(df_silver_pib.limit(10))

# ── Silver PIB Componentes ────────────────────────────────────────────────────
if df_comp_raw is not None:
    df_silver_comp = standardize_pib_multi(df_comp_raw, _MAP_COMP)
    save_delta(df_silver_comp, "silver_pib_componentes")
    display(df_silver_comp.limit(10))